In [3]:
import pandas as pd
import numpy as np
import pickle
import warnings
import optuna
import shap
warnings.filterwarnings('ignore')

from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
import category_encoders as ce

# ─────────────────────────────────────────────────────────────
# STEP 1 — Load and prepare data
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("STEP 1 — Loading and preparing data")
print("=" * 60)

# CHANGE THIS PATH TO YOUR NEW EXCEL FILE
FILE_PATH = r'C:\Users\MSI\Desktop\PFE\PHASE 2\seed_family_code\NEW FIXES\microrna_with_families_all.xlsx'
df = pd.read_excel(FILE_PATH)

# Clean up parasite names
df['parasite'] = df['parasite'].str.replace(r'\s+', '', regex=True)

# Drop columns the model won't use. 
# (mirbase_accession is safely ignored because we don't put it in the feature lists later)
# ── Build lookup dicts BEFORE dropping microrna / microrna_group_simplified ──
# These are used at prediction time in the Streamlit app so the user can input
# either the full miRNA name (e.g. hsa-miR-155-5p) OR the accession number
# (e.g. MIMAT0000646) and both resolve to the correct seed_family automatically.
df_lookup = df.copy()  # keep a copy with all columns before dropping

# miRNA name → {microrna_group_simplified, seed_family, mirbase_accession}
mirna_lookup = (
    df_lookup[['microrna', 'microrna_group_simplified', 'seed_family', 'mirbase_accession']]
    .drop_duplicates('microrna')
    .set_index('microrna')
    .to_dict('index')
)

# accession number → {microrna, microrna_group_simplified, seed_family}
# Only for rows where accession is known (not NaN / not_found)
accession_lookup = (
    df_lookup[['mirbase_accession', 'microrna', 'microrna_group_simplified', 'seed_family']]
    .dropna(subset=['mirbase_accession'])
    .drop_duplicates('mirbase_accession')
    .set_index('mirbase_accession')
    .to_dict('index')
)

print(f"miRNA lookup     : {len(mirna_lookup)} unique miRNA names")
print(f"Accession lookup : {len(accession_lookup)} unique accession numbers")

cols_to_drop =['microrna', 'infection', 'microrna_group_simplified']
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# Replace both 'not_broadly_conserved' and 'not_found' with NaN (missing value)
# LightGBM will automatically learn to bypass these missing values using other features!
df['seed_family'] = df['seed_family'].replace(['not_broadly_conserved', 'not_found'], np.nan)

# Create interaction feature: parasite × cell type
df['parasite_celltype'] = df['parasite'].astype(str) + '_' + df['cell type'].astype(str)

print(f"Total rows            : {len(df)}")
print(f"Target balance        :\n{df['is_upregulated'].value_counts().to_string()}")


# ─────────────────────────────────────────────────────────────
# STEP 2 — Features and target
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2 — Features and target")
print("=" * 60)

TARGET = 'is_upregulated'

# Categorical columns (Text)
CAT_COLS =['parasite', 'organism', 'cell type', 'seed_family', 'parasite_celltype']

# Numeric columns (Numbers) - Added family_conservation here!
NUM_COLS =['time', 'is_conserved', 'family_conservation']

X = df[CAT_COLS + NUM_COLS]
y = df[TARGET]

print(f"Categorical   : {CAT_COLS}")
print(f"Numeric       : {NUM_COLS}")
print(f"Total features: {len(CAT_COLS) + len(NUM_COLS)}")


# ─────────────────────────────────────────────────────────────
# STEP 3 — Optuna Hyperparameter Tuning
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3 — Finding best parameters for NEW dataset (Optuna)")
print("=" * 60)

def objective(trial):
    # Parameters to search
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 10, 50),
        'min_child_samples': trial.suggest_int('min_child_samples', 2, 20),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'class_weight': 'balanced',
        'random_state': 42,
        'verbose': -1
    }
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    pipe_opt = Pipeline([
        ('encoder', ce.TargetEncoder(cols=CAT_COLS, smoothing=1.0, handle_missing='value')),
        ('model', LGBMClassifier(**params))
    ])
    
    score = cross_val_score(pipe_opt, X, y, cv=cv, scoring='roc_auc').mean()
    return score

# Run the search (30 trials is a good balance of speed and accuracy)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize')
print("Running 30 trials to find the best LightGBM parameters... Please wait.")
study.optimize(objective, n_trials=30)

BEST_PARAMS = study.best_params
BEST_PARAMS['class_weight'] = 'balanced'
BEST_PARAMS['random_state'] = 42
BEST_PARAMS['verbose'] = -1

print(f"Best ROC-AUC found: {study.best_value:.4f}")
print("Best Parameters:")
for k, v in BEST_PARAMS.items():
    print(f"  {k}: {v}")


# ─────────────────────────────────────────────────────────────
# STEP 4 — Cross-Validation with New Best Params
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4 — Cross-Validation Verification")
print("=" * 60)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe = Pipeline([
    ('encoder', ce.TargetEncoder(cols=CAT_COLS, smoothing=1.0, handle_missing='value')),
    ('model',   LGBMClassifier(**BEST_PARAMS))
])

auc = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc')
acc = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy')
f1  = cross_val_score(pipe, X, y, cv=cv, scoring='f1')

print(f"ROC-AUC : {auc.mean():.3f} ± {auc.std():.3f}")
print(f"Accuracy: {acc.mean():.3f} ± {acc.std():.3f}")
print(f"F1      : {f1.mean():.3f} ± {f1.std():.3f}")
print(f"AUC per fold: {[round(x, 3) for x in auc.tolist()]}")


# ─────────────────────────────────────────────────────────────
# STEP 5 — Train final model on all data
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5 — Training final model on all data")
print("=" * 60)

pipe.fit(X, y)
print("Done.")


# ─────────────────────────────────────────────────────────────
# STEP 6 — Permutation Feature Importance & SHAP
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6 — Feature Importance (Checking family_conservation)")
print("=" * 60)

X_enc = pipe.named_steps['encoder'].transform(X)

# Permutation importance
perm  = permutation_importance(
    pipe.named_steps['model'],
    X_enc, y, n_repeats=30, random_state=42, scoring='roc_auc'
)

perm_df = pd.DataFrame({
    'feature':    X.columns.tolist(),
    'importance': perm.importances_mean,
    'std':        perm.importances_std
}).sort_values('importance', ascending=False)

print(f"\n{'Feature':<25} {'Importance':>10} {'Std':>8}")
print("-" * 47)
for _, row in perm_df.iterrows():
    bar  = '█' * max(0, int(row['importance'] * 60))
    sign = '+' if row['importance'] >= 0 else ''
    print(f"  {row['feature']:<23} {sign}{row['importance']:.4f} ± {row['std']:.4f}  {bar}")

# Generate SHAP explainer
explainer = shap.TreeExplainer(pipe.named_steps['model'])


# ─────────────────────────────────────────────────────────────
# STEP 7 — Save model bundle
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7 — Saving model bundle")
print("=" * 60)

model_bundle = {
    'model':     pipe,
    'encoder':   pipe.named_steps['encoder'],
    'lgbm':      pipe.named_steps['model'],
    'explainer': explainer,  # SHAP explainer is now properly defined!
    'metrics': {
        'auc_mean':   round(auc.mean(), 3),
        'auc_std':    round(auc.std(),  3),
        'acc_mean':   round(acc.mean(), 3),
        'acc_std':    round(acc.std(),  3),
        'f1_mean':    round(f1.mean(),  3),
        'f1_std':     round(f1.std(),   3),
        'auc_folds':  [round(x, 3) for x in auc.tolist()],
        'best_params': BEST_PARAMS,
        'n_train':    len(X),
        'feature_importance': perm_df.to_dict('records'),
    },
    'options': {
        'parasite':      sorted(df['parasite'].dropna().unique().tolist()),
        'organism':      sorted(df['organism'].dropna().unique().tolist()),
        'cell_type':     sorted(df['cell type'].dropna().unique().tolist()),
        'time':          sorted(df['time'].dropna().unique().tolist()),
        'seed_families': sorted(df['seed_family'].dropna().unique().tolist()),
        # Added family_conservation to options for Streamlit usage
        'family_conservation': sorted(df['family_conservation'].dropna().unique().tolist()) 
    },
    'cat_cols':      CAT_COLS,
    'feature_names': list(X.columns),
    # Lookup dicts for Streamlit app — user can input miRNA name OR accession number
    'mirna_lookup':      mirna_lookup,      # miRNA name  → {group, seed_family, accession}
    'accession_lookup':  accession_lookup,  # accession   → {microrna, group, seed_family}
}

BUNDLE_PATH = r'C:\Users\MSI\Desktop\PFE\PHASE 2\seed_family_code\NEW FIXES\Model206_ALL_model.pkl'
with open(BUNDLE_PATH, 'wb') as f:
    pickle.dump(model_bundle, f)
print(f"Saved: {BUNDLE_PATH}")

STEP 1 — Loading and preparing data
miRNA lookup     : 115 unique miRNA names
Accession lookup : 98 unique accession numbers
Total rows            : 206
Target balance        :
is_upregulated
1    103
0    103

STEP 2 — Features and target
Categorical   : ['parasite', 'organism', 'cell type', 'seed_family', 'parasite_celltype']
Numeric       : ['time', 'is_conserved', 'family_conservation']
Total features: 8

STEP 3 — Finding best parameters for NEW dataset (Optuna)
Running 30 trials to find the best LightGBM parameters... Please wait.
Best ROC-AUC found: 0.9275
Best Parameters:
  n_estimators: 380
  max_depth: 7
  learning_rate: 0.01759016072902054
  num_leaves: 25
  min_child_samples: 4
  subsample: 0.9302716477616597
  colsample_bytree: 0.532296323819081
  reg_alpha: 2.9871276967074894e-05
  reg_lambda: 0.2125970800306679
  class_weight: balanced
  random_state: 42
  verbose: -1

STEP 4 — Cross-Validation Verification
ROC-AUC : 0.928 ± 0.039
Accuracy: 0.845 ± 0.044
F1      : 0.840 ±